# Runebound Chess — test in Colab

This runs the Flask app inside Colab and exposes it with a temporary public URL, so you can test it (and play with a friend) without installing anything locally.

**Use the Hugging Face backend here, not Ollama.** Colab's free tier doesn't reliably support running a background Ollama server alongside your notebook, and you don't need a GPU for this app anyway — the LLM call goes out over HTTPS to Hugging Face's cloud. You'll need a free HF token: https://huggingface.co/settings/tokens (fine-grained, with "Make calls to Inference Providers" permission).

**Steps:** run the cells top to bottom. The last cell prints a public link — open it, or share it.

## 1. Upload the project
Upload `rpg-chess.zip` (the file you downloaded) using the file picker below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick rpg-chess.zip in the dialog

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

%cd rpg-chess
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt pyngrok

## 3. Configure the LLM backend (Hugging Face)

Paste your token when prompted. It's read into memory only, never written to a file or printed.

In [ ]:
import os
from getpass import getpass

os.environ['LLM_PROVIDER'] = 'huggingface'
os.environ['HF_TOKEN'] = getpass('Paste your Hugging Face token: ')
os.environ['HF_MODEL'] = 'meta-llama/Llama-3.2-3B-Instruct'  # swap for any HF chat model you like
print('Configured.')

## 4. Run the app in the background

In [ ]:
import subprocess, time, sys

server_process = subprocess.Popen(
    [sys.executable, 'server.py'],
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(3)  # give Flask a moment to bind the port
print('Flask app started, pid =', server_process.pid)

## 5. Expose it with a public URL

Optional but recommended: a free ngrok account gives more stable tunnels than the anonymous ones. Get an authtoken at https://dashboard.ngrok.com/get-started/your-authtoken and paste it below — or just leave it blank to use an anonymous tunnel (works fine for quick tests, just rate-limited).

In [ ]:
from pyngrok import ngrok, conf

token = getpass('ngrok authtoken (press Enter to skip): ')
if token.strip():
    conf.get_default().auth_token = token.strip()

public_url = ngrok.connect(5000)
print('Your game is live at:', public_url)

Open the printed URL in your browser (or send it to whoever you're playing with). Both players share this one link and take turns making moves.

## 6. Check logs / stop the server
Run these cells if something looks wrong, or when you're done.

In [ ]:
# Peek at the last lines of Flask's log if you want to debug something
print(server_process.stdout.readline())

In [ ]:
# Shut everything down
ngrok.disconnect(public_url.public_url)
server_process.terminate()
print('Stopped.')

---
### Notes
- This is for **testing only** — Colab sessions time out (usually a few hours of inactivity, or ~12h max), so it's not a real deployment. See the project's `README.md` for actual Docker / cloud deployment steps once you're happy with it.
- If you'd rather test with a self-hosted Ollama model instead of Hugging Face, that also technically works in Colab (`!curl -fsSL https://ollama.com/install.sh | sh`, then run `ollama serve` and `ollama pull llama3.2` in background subprocesses, and set `LLM_PROVIDER=ollama`), but it's slower to set up here and Colab's free GPUs aren't guaranteed to be available — Hugging Face is the simpler path for a quick test.